# 03 — Evaluation

Runs all baselines and HaluGuard on the CodeHaluEval **test** split.
Produces hallucination rate, pass rate, and reduction ratio tables.

**Prerequisites:**
- `data/triplets.jsonl` from notebook 01
- `checkpoints/hccs_best.pt` from notebook 02

**Methods compared:**
1. No context (lower bound)
2. BM25 retrieval
3. Full context (upper-bound context)
4. CodeBERT cosine similarity (traditional RAG)
5. **HaluGuard** (HCCS + type router + EFL)

**Runtime:** ~10 hours on T4 for the full test set

## Setup

In [ ]:
!pip install torch transformers datasets rank-bm25 tqdm --quiet

In [ ]:
import sys, os, json
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/drive/MyDrive/HaluGuard'
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
!pip install -e '.[dev]' -q

from notebooks.utils import check_gpu, get_drive_path, append_jsonl, count_jsonl
DEVICE = check_gpu()

DRIVE_DIR   = get_drive_path('HaluGuard')
RESULTS_DIR = DRIVE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH   = DRIVE_DIR / 'checkpoints' / 'hccs_best.pt'
print(f'Checkpoint exists: {CKPT_PATH.exists()}')

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

from haluguard.chunker import chunk_repo
from haluguard.efl import execute_code
from haluguard.evaluate import (
    compute_metrics_table,
    compute_per_type_hr,
    compute_reduction_ratio,
    evaluate_baseline,
    evaluate_haluguard,
)
from haluguard.hccs import HCCSScorer, batch_embed, embed_code
from haluguard.pipeline import HaluGuardPipeline

## 1. Load Dataset and Models

In [ ]:
import ast
from datasets import load_dataset
from haluguard.hccs import HallucinationType

# Load the test split — same dataset as notebook 01
ds = load_dataset('Yuchen111/CodeHaluEval', split='train')
# Use the last 20% as a held-out test set (consistent with notebook 01 using first 500)
TEST_START = 500
ds_test = ds.select(range(TEST_START, len(ds)))
print(f'Test set: {len(ds_test)} samples (tasks {TEST_START}–{len(ds)})')

# Reuse the same HALU_TYPE_MAP and extract helpers from notebook 01
HALU_TYPE_MAP = {
    'physical_constraint_hallucination': HallucinationType.LOGIC,
    'logic_breakdown':                   HallucinationType.LOGIC,
    'logic_deviation':                   HallucinationType.LOGIC,
    'calculate_boundary_hallucination':  HallucinationType.LOGIC,
    'identification_hallucination':      HallucinationType.NAMING,
    'structural_access_hallucination':   HallucinationType.MAPPING,
    'data_compliance_hallucination':     HallucinationType.MAPPING,
    'external_source_hallucination':     HallucinationType.RESOURCE,
}


def _parse_value(s: str):
    s = s.strip()
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        pass
    lines = s.splitlines()
    if len(lines) == 1:
        tokens = s.split()
        if len(tokens) == 1:
            for cast in (int, float):
                try: return cast(s)
                except ValueError: pass
            return s
        parsed = []
        for t in tokens:
            for cast in (int, float):
                try: parsed.append(cast(t)); break
                except ValueError: pass
            else:
                parsed.append(t)
        return parsed
    return [_parse_value(line) for line in lines]


def build_test_code(fn_name: str, input_str: str, output_str: str) -> str:
    fn = (fn_name or 'solve').strip()
    input_val    = _parse_value(input_str)
    expected_val = _parse_value(output_str)
    args_repr    = ', '.join(repr(v) for v in input_val) if isinstance(input_val, list) else repr(input_val)
    return (
        f"_result   = {fn}({args_repr})\n"
        f"_expected = {repr(expected_val)}\n"
        f"assert _result == _expected, f'Expected {{_expected!r}}, got {{_result!r}}'"
    )


def extract_task(sample: dict) -> dict:
    solutions = sample.get('solutions') or []
    if isinstance(solutions, str):
        solutions = [solutions]
    repo_files = {f'solution_{i}.py': sol for i, sol in enumerate(solutions) if sol}
    if not repo_files:
        repo_files = {'placeholder.py': '# No reference solution provided'}
    fn_name = sample.get('fn_name') or 'solve'
    return {
        'task_id':            str(sample.get('id', sample.get('task_id', '?'))),
        'query':              sample['question'],
        'repo_files':         repo_files,
        'test_code':          build_test_code(fn_name, sample.get('input', ''), sample.get('output', '')),
        'fn_name':            fn_name,
        'hallucination_type': HALU_TYPE_MAP.get(
            (sample.get('halu_type') or '').lower().replace(' ', '_')
        ),
    }


print('extract_task defined. Sample task:')
t = extract_task(ds_test[0])
print('  task_id:', t['task_id'], '| fn_name:', t['fn_name'], '| halu_type:', t['hallucination_type'])

In [ ]:
# Load CodeBERT
ENCODER_NAME = 'microsoft/codebert-base'
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
encoder   = AutoModel.from_pretrained(ENCODER_NAME).to(DEVICE)
encoder.eval()
print('CodeBERT loaded.')

# Load trained HCCS scorer
scorer = HCCSScorer.load(CKPT_PATH).to(DEVICE)
scorer.eval()
print('HCCS scorer loaded.')

In [ ]:
# Load DeepSeek-Coder (same as notebook 01)
from transformers import AutoModelForCausalLM

MODEL_NAME = 'deepseek-ai/deepseek-coder-1.3b-instruct'
gen_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
gen_model.eval()
print(f'DeepSeek-Coder loaded on {next(gen_model.parameters()).device}')


def generate_code(prompt: str, max_new_tokens: int = 256) -> str:
    """Generate code from a prompt using DeepSeek-Coder-1.3B."""
    inputs = gen_tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=1024
    ).to(DEVICE)
    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.95,
            pad_token_id=gen_tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return gen_tokenizer.decode(new_tokens, skip_special_tokens=True)

## 2. Retrieval Helpers

In [ ]:
TOP_K = 5


def bm25_retrieval_fn(query: str, chunks: list, top_k: int = TOP_K) -> list:
    """BM25 keyword retrieval — selects top-k chunks by token overlap."""
    if not chunks:
        return []
    tokenized = [c.lower().split() for c in chunks]
    bm25 = BM25Okapi(tokenized)
    scores = bm25.get_scores(query.lower().split())
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [chunks[i] for i in top_idx]


def cosine_retrieval_fn(query: str, chunks: list, top_k: int = TOP_K) -> list:
    """CodeBERT cosine similarity retrieval — traditional RAG baseline."""
    if not chunks:
        return []
    q_emb  = embed_code(query, tokenizer, encoder, device=DEVICE)
    c_embs = batch_embed(chunks, tokenizer, encoder, device=DEVICE)
    q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-8)
    c_norms = c_embs / (np.linalg.norm(c_embs, axis=1, keepdims=True) + 1e-8)
    sims = c_norms @ q_norm
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [chunks[i] for i in top_idx]


print('Retrieval helpers ready.')

## 3. Run Evaluations

In [ ]:
# --- Baseline 1: No context ---
NO_CTX_PATH = RESULTS_DIR / 'results_no_context.jsonl'
done_ids = set()
if NO_CTX_PATH.exists():
    with NO_CTX_PATH.open() as f:
        done_ids = {json.loads(l)['task_id'] for l in f if l.strip()}

remaining = [t for t in tasks if t['task_id'] not in done_ids]
print(f'No-context: {len(done_ids)} done, {len(remaining)} remaining')

for task in tqdm(remaining, desc='No context'):
    results = evaluate_baseline('no_context', [task], generate_code)
    append_jsonl(results[0], NO_CTX_PATH)

print(f'Total: {count_jsonl(NO_CTX_PATH)} results')

In [ ]:
# --- Baseline 2: BM25 ---
BM25_PATH = RESULTS_DIR / 'results_bm25.jsonl'
done_ids = set()
if BM25_PATH.exists():
    with BM25_PATH.open() as f:
        done_ids = {json.loads(l)['task_id'] for l in f if l.strip()}

remaining = [t for t in tasks if t['task_id'] not in done_ids]
print(f'BM25: {len(done_ids)} done, {len(remaining)} remaining')

for task in tqdm(remaining, desc='BM25'):
    results = evaluate_baseline('bm25', [task], generate_code,
                                retrieval_fn=bm25_retrieval_fn)
    append_jsonl(results[0], BM25_PATH)

print(f'Total: {count_jsonl(BM25_PATH)} results')

In [ ]:
# --- Baseline 3: Full context ---
FULL_CTX_PATH = RESULTS_DIR / 'results_full_context.jsonl'
done_ids = set()
if FULL_CTX_PATH.exists():
    with FULL_CTX_PATH.open() as f:
        done_ids = {json.loads(l)['task_id'] for l in f if l.strip()}

remaining = [t for t in tasks if t['task_id'] not in done_ids]
print(f'Full context: {len(done_ids)} done, {len(remaining)} remaining')

for task in tqdm(remaining, desc='Full context'):
    results = evaluate_baseline('full_context', [task], generate_code)
    append_jsonl(results[0], FULL_CTX_PATH)

print(f'Total: {count_jsonl(FULL_CTX_PATH)} results')

In [ ]:
# --- Baseline 4: CodeBERT cosine similarity ---
CODEBERT_PATH = RESULTS_DIR / 'results_codebert_sim.jsonl'
done_ids = set()
if CODEBERT_PATH.exists():
    with CODEBERT_PATH.open() as f:
        done_ids = {json.loads(l)['task_id'] for l in f if l.strip()}

remaining = [t for t in tasks if t['task_id'] not in done_ids]
print(f'CodeBERT sim: {len(done_ids)} done, {len(remaining)} remaining')

for task in tqdm(remaining, desc='CodeBERT sim'):
    results = evaluate_baseline('codebert_similarity', [task], generate_code,
                                retrieval_fn=cosine_retrieval_fn)
    append_jsonl(results[0], CODEBERT_PATH)

print(f'Total: {count_jsonl(CODEBERT_PATH)} results')

In [ ]:
# --- HaluGuard: Full pipeline ---
HALUGUARD_PATH = RESULTS_DIR / 'results_haluguard.jsonl'
done_ids = set()
if HALUGUARD_PATH.exists():
    with HALUGUARD_PATH.open() as f:
        done_ids = {json.loads(l)['task_id'] for l in f if l.strip()}

remaining = [t for t in tasks if t['task_id'] not in done_ids]
print(f'HaluGuard: {len(done_ids)} done, {len(remaining)} remaining')

pipeline = HaluGuardPipeline(
    scorer=scorer,
    tokenizer=tokenizer,
    encoder=encoder,
    top_k=TOP_K,
    device=DEVICE,
)

for task in tqdm(remaining, desc='HaluGuard'):
    results = evaluate_haluguard([task], pipeline, generate_fn=generate_code)
    append_jsonl(results[0], HALUGUARD_PATH)

print(f'Total: {count_jsonl(HALUGUARD_PATH)} results')

## 4. Results Table

In [ ]:
def load_jsonl(path):
    from pathlib import Path
    p = Path(path)
    if not p.exists(): return []
    with p.open() as f:
        return [json.loads(l) for l in f if l.strip()]

results_by_method = {
    'no_context':   load_jsonl(NO_CTX_PATH),
    'bm25':         load_jsonl(BM25_PATH),
    'full_context': load_jsonl(FULL_CTX_PATH),
    'codebert_sim': load_jsonl(CODEBERT_PATH),
    'haluguard':    load_jsonl(HALUGUARD_PATH),
}
results_by_method = {k: v for k, v in results_by_method.items() if v}

if results_by_method:
    table = compute_metrics_table(results_by_method, baseline_key='no_context')
    print(f'{"Method":<22} {"HR":>6}  {"Pass%":>6}  {"Reduction":>9}')
    print('-' * 50)
    for row in table:
        print(f'{row["method"]:<22}{row["hr"]:>6.3f}  {row["pass_rate"]:>6.3f}  {row["reduction_ratio"]:>9.3f}')
else:
    print('No results yet — run evaluation cells above.')

In [ ]:
# Per-type breakdown: HaluGuard vs BM25
if 'haluguard' in results_by_method and 'bm25' in results_by_method:
    print(f'{"Type":<12} {"BM25":>8}  {"HaluGuard":>10}  {"Reduction":>10}')
    print('-' * 46)
    bm25_by_type = compute_per_type_hr(results_by_method['bm25'])
    hg_by_type   = compute_per_type_hr(results_by_method['haluguard'])
    for t in ['resource', 'naming', 'mapping', 'logic']:
        b = bm25_by_type.get(t, float('nan'))
        h = hg_by_type.get(t, float('nan'))
        try:
            r = compute_reduction_ratio(b, h)
        except Exception:
            r = float('nan')
        print(f'{t:<12} {b:>8.3f}  {h:>10.3f}  {r:>10.3f}')

In [ ]:
os.chdir(REPO_DIR)
!git add -A
!git commit -m 'feat: add evaluation results'
!git push